# Disease and Retail Availability Data Cleaning

This notebook cleans the California COPD prevalence data from the CDC API and the retail availability of electronic smoking devices dataset. It preserves the raw data, writes cleaned interim files, merges the datasets at the county level, and exports a final analysis-ready CSV.

In [ ]:
from pathlib import Path

import pandas as pd
import requests

pd.set_option("display.max_columns", 50)

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != "disease_data":
    PROJECT_DIR = PROJECT_DIR / "disease_data"

RAW_DIR = PROJECT_DIR / "data" / "raw"
INTERIM_DIR = PROJECT_DIR / "data" / "interim"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RETAIL_RAW_FILE = RAW_DIR / "retail-availability-of-electronic-smoking-devices-by-county.csv"
COPD_INTERIM_FILE = INTERIM_DIR / "copd_ca_county_clean.csv"
RETAIL_INTERIM_FILE = INTERIM_DIR / "retail_esd_availability_clean.csv"
MERGED_PROCESSED_FILE = PROCESSED_DIR / "ca_county_copd_retail_esd_merged.csv"

In [ ]:
def normalize_county_name(value):
    """Standardize county names for joining while keeping readable labels elsewhere."""
    if pd.isna(value):
        return pd.NA

    text = str(value).strip()
    text = " ".join(text.split())

    if text.lower().endswith(" county"):
        text = text[:-7].strip()

    return text.title()


def summarize_missing(df):
    return df.isna().sum().rename("missing_count").to_frame()

## Load and Clean COPD Data

In [ ]:
copd_url = "https://data.cdc.gov/resource/i46a-9kgh.json"
copd_params = {
    "$limit": 5000,
    "$select": "stateabbr,countyname,countyfips,COPD_AdjPrev",
    "stateabbr": "CA",
}

response = requests.get(copd_url, params=copd_params, timeout=30)
response.raise_for_status()

copd_raw = pd.DataFrame(response.json())
copd_raw.head()

In [ ]:
copd_clean = (
    copd_raw.rename(
        columns={
            "stateabbr": "state",
            "countyname": "county",
            "countyfips": "county_fips",
            "COPD_AdjPrev": "copd_adjusted_prevalence_pct",
        }
    )
    .assign(
        county=lambda df: df["county"].map(normalize_county_name),
        county_fips=lambda df: df["county_fips"].astype("string").str.zfill(5),
        copd_adjusted_prevalence_pct=lambda df: pd.to_numeric(
            df["copd_adjusted_prevalence_pct"], errors="coerce"
        ),
    )
    .dropna(subset=["county", "county_fips", "copd_adjusted_prevalence_pct"])
    .drop_duplicates(subset=["county_fips"])
    .sort_values("county")
    .reset_index(drop=True)
)

copd_clean.to_csv(COPD_INTERIM_FILE, index=False)
copd_clean.head()

## Load and Clean Retail Availability Data

In [ ]:
retail_raw = pd.read_csv(RETAIL_RAW_FILE)
retail_raw.head()

In [ ]:
non_county_rows = {"Berkeley", "Long Beach", "Pasadena", "Statewide"}

retail_clean = (
    retail_raw.rename(
        columns={
            "County": "county",
            "Year": "year",
            "Percentage": "retail_esd_availability_pct",
        }
    )
    .assign(
        county=lambda df: df["county"].map(normalize_county_name),
        year=lambda df: pd.to_numeric(df["year"], errors="coerce").astype("Int64"),
        retail_esd_availability_pct=lambda df: pd.to_numeric(
            df["retail_esd_availability_pct"].replace({"*": pd.NA}), errors="coerce"
        ),
    )
)

retail_clean = (
    retail_clean.loc[~retail_clean["county"].isin(non_county_rows)]
    .dropna(subset=["county", "year"])
    .drop_duplicates(subset=["county", "year"])
    .sort_values(["county", "year"])
    .reset_index(drop=True)
)

retail_clean.to_csv(RETAIL_INTERIM_FILE, index=False)
retail_clean.head()

## Merge and Final Cleaning

In [ ]:
merged = retail_clean.merge(copd_clean, on="county", how="left", validate="many_to_one")

merged = (
    merged.assign(
        retail_esd_availability_pct=lambda df: df["retail_esd_availability_pct"].round(3),
        copd_adjusted_prevalence_pct=lambda df: df["copd_adjusted_prevalence_pct"].round(2),
        retail_esd_availability_suppressed=lambda df: df["retail_esd_availability_pct"].isna(),
    )
    [
        [
            "state",
            "county",
            "county_fips",
            "year",
            "retail_esd_availability_pct",
            "retail_esd_availability_suppressed",
            "copd_adjusted_prevalence_pct",
        ]
    ]
    .sort_values(["county", "year"])
    .reset_index(drop=True)
)

merged.to_csv(MERGED_PROCESSED_FILE, index=False)
merged.head()

In [ ]:
quality_summary = pd.DataFrame(
    {
        "dataset": ["copd_clean", "retail_clean", "merged"],
        "rows": [len(copd_clean), len(retail_clean), len(merged)],
        "counties": [
            copd_clean["county"].nunique(),
            retail_clean["county"].nunique(),
            merged["county"].nunique(),
        ],
    }
)

unmatched_retail_counties = sorted(
    set(retail_clean["county"].dropna()) - set(copd_clean["county"].dropna())
)

print("Quality summary")
display(quality_summary)

print("Missing values in final merged dataset")
display(summarize_missing(merged))

print("Retail counties without COPD match:", unmatched_retail_counties)
print("Files written:")
print(f"- {COPD_INTERIM_FILE}")
print(f"- {RETAIL_INTERIM_FILE}")
print(f"- {MERGED_PROCESSED_FILE}")